In [ ]:
!pip install --upgrade --force-reinstall numpy==2.0.2 scipy==1.14.0 pandas==2.2.2 matplotlib==3.9.2 seaborn==0.13.2 requests==2.32.4 pillow==11.1.0 google-generativeai openai anthropic -q

import os
import json
import re
import time
import uuid
import math
import statistics
import random
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from scipy import stats as scipy_stats
except ImportError:
    scipy_stats = None

from openai import OpenAI
from anthropic import Anthropic

try:
    import google.generativeai as genai
    HAS_GEMINI = True
except ImportError:
    HAS_GEMINI = False

OUTPUT_DIR = "benchmark_output"
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")
RESULTS_FILE = os.path.join(OUTPUT_DIR, "results_backup.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# OPTIMIZED CONFIGURATION FOR RATE LIMITING
# ============================================================================
# Testing 6 models (4 OpenAI + 1 Anthropic + 1 Google)
MODELS = [
         # OpenAI fast baseline
    "gpt-4o",                # OpenAI flagship
                # OpenAI next-gen (verify model name exists!)
    "gpt-4o-mini",                 # OpenAI next-gen (verify model name exists!)
    "claude-3-5-sonnet-20241022",  # Anthropic flagship
    "gemini-2.5-flash",      # Google flagship
]

# Reduced from 5 runs to 3 (still statistically valid with n=3)
N_RUNS = 3

# Reduced scenarios from 40 to 18 (6 per difficulty level)
# This maintains difficulty distribution while reducing load
SCENARIO_SUBSET_SIZE = 6  # per difficulty level

TEMPERATURE = 0.2
MAX_TOKENS = 300
MAX_TURNS = 4
SEED = 42
WORKERS = 1  # Keep at 1 to avoid rate limit bursts

# Add delays between API calls
API_DELAY_SECONDS = 2.0  # 2 second delay between calls

VERSION = "3.0-workshop"
NAME = "LTA-E: Long Trajectory Agentic Execution Benchmark (Workshop Edition)"
FOCUS = "Multi-turn agentic performance evaluation on restaurant booking tasks"

print("\n" + "="*80)
print("LTA-E Benchmark v{0}".format(VERSION))
print(NAME)
print("="*80)
print("WORKSHOP-OPTIMIZED CONFIGURATION:")
print("  Models: {0} (4 OpenAI + 1 Anthropic + 1 Google)".format(len(MODELS)))
print("  Scenarios per difficulty: {0} (18 total, reduced from 40)".format(SCENARIO_SUBSET_SIZE))
print("  Runs per scenario: {0} (reduced from 5)".format(N_RUNS))
print("  Total API calls: {0} (reduced from 1,400)".format(len(MODELS) * SCENARIO_SUBSET_SIZE * 3 * N_RUNS))
print("  API delay: {0}s between calls".format(API_DELAY_SECONDS))
print("  Estimated runtime: 45-65 minutes")
print("="*80 + "\n")

# Full scenario pool - we'll sample from this
ALL_SCENARIOS = [
    {"id": "s1_easy", "text": "Book a table for 2 at an Italian restaurant in New York on Friday at 8 PM.", "expected": {"people": 2, "cuisine": "italian", "city": "new york", "day": "friday", "time": "8 PM"}, "difficulty": "easy"},
    {"id": "s2_easy", "text": "Book a table for 4 at an Indian restaurant in San Jose on Monday at 6 PM.", "expected": {"people": 4, "cuisine": "indian", "city": "san jose", "day": "monday", "time": "6 PM"}, "difficulty": "easy"},
    {"id": "s3_easy", "text": "Book a table for 6 at an Italian restaurant in New Jersey on Friday at 10 PM.", "expected": {"people": 6, "cuisine": "italian", "city": "new jersey", "day": "friday", "time": "10 PM"}, "difficulty": "easy"},
    {"id": "s4_easy", "text": "Book a table for 2 at a sushi restaurant in San Francisco on Saturday at 7 PM.", "expected": {"people": 2, "cuisine": "sushi", "city": "san francisco", "day": "saturday", "time": "7 PM"}, "difficulty": "easy"},
    {"id": "s5_easy", "text": "Reserve a table for 4 at a Mexican restaurant in Austin on Thursday at 6 PM.", "expected": {"people": 4, "cuisine": "mexican", "city": "austin", "day": "thursday", "time": "6 PM"}, "difficulty": "easy"},
    {"id": "s6_easy", "text": "Book dinner for 3 at a Chinese restaurant in Chicago on Monday at 8 PM.", "expected": {"people": 3, "cuisine": "chinese", "city": "chicago", "day": "monday", "time": "8 PM"}, "difficulty": "easy"},
    {"id": "s7_easy", "text": "Make a reservation for 5 at an Indian restaurant in Seattle on Sunday at 7:30 PM.", "expected": {"people": 5, "cuisine": "indian", "city": "seattle", "day": "sunday", "time": "7:30 PM"}, "difficulty": "easy"},
    {"id": "s8_easy", "text": "Book a table for 2 at a pizza place in Los Angeles on Tuesday at 6:45 PM.", "expected": {"people": 2, "cuisine": "pizza", "city": "los angeles", "day": "tuesday", "time": "6:45 PM"}, "difficulty": "easy"},
    {"id": "s9_easy", "text": "Table for 3 at a Thai restaurant in Miami on Wednesday at 7 PM.", "expected": {"people": 3, "cuisine": "thai", "city": "miami", "day": "wednesday", "time": "7 PM"}, "difficulty": "easy"},
    {"id": "s10_easy", "text": "Reserve 4 seats at a French bistro in Boston on Saturday at 8 PM.", "expected": {"people": 4, "cuisine": "french", "city": "boston", "day": "saturday", "time": "8 PM"}, "difficulty": "easy"},
    {"id": "s11_easy", "text": "Book 2 at a Greek restaurant in Phoenix on Thursday at 6:30 PM.", "expected": {"people": 2, "cuisine": "greek", "city": "phoenix", "day": "thursday", "time": "6:30 PM"}, "difficulty": "easy"},
    {"id": "s12_easy", "text": "Make a reservation for 5 at a Portuguese restaurant in Denver on Friday at 7 PM.", "expected": {"people": 5, "cuisine": "portuguese", "city": "denver", "day": "friday", "time": "7 PM"}, "difficulty": "easy"},

    {"id": "s1_medium", "text": "I need a reservation for a plant-based restaurant. Make it for 4 people in Portland next Tuesday around 7 PM.", "expected": {"people": 4, "cuisine": "vegan", "city": "portland", "day": "tuesday", "time": "7 PM"}, "difficulty": "medium"},
    {"id": "s2_medium", "text": "Can you book a Korean BBQ spot for my team of 5 in Denver this coming Friday evening?", "expected": {"people": 5, "cuisine": "korean", "city": "denver", "day": "friday", "time": "7 PM"}, "difficulty": "medium"},
    {"id": "s3_medium", "text": "I'm looking for a Mediterranean place in Philadelphia for 3 people, preferably next Monday night.", "expected": {"people": 3, "cuisine": "mediterranean", "city": "philadelphia", "day": "monday", "time": "7 PM"}, "difficulty": "medium"},
    {"id": "s4_medium", "text": "Find me a steakhouse in Dallas for a party of 8, sometime next weekend in the evening.", "expected": {"people": 8, "cuisine": "steakhouse", "city": "dallas", "day": "saturday", "time": "7 PM"}, "difficulty": "medium"},
    {"id": "s5_medium", "text": "Book a ramen place for 2 in San Diego, next Thursday after work hours.", "expected": {"people": 2, "cuisine": "ramen", "city": "san diego", "day": "thursday", "time": "6 PM"}, "difficulty": "medium"},
    {"id": "s6_medium", "text": "I want to try Ethiopian food in Washington DC with 4 friends next Saturday at dinner time.", "expected": {"people": 5, "cuisine": "ethiopian", "city": "washington dc", "day": "saturday", "time": "7 PM"}, "difficulty": "medium"},
    {"id": "s7_medium", "text": "Reserve a tapas restaurant in San Antonio for 4 people, next week on Wednesday around 8 PM.", "expected": {"people": 4, "cuisine": "tapas", "city": "san antonio", "day": "wednesday", "time": "8 PM"}, "difficulty": "medium"},
    {"id": "s8_medium", "text": "Can you find a Vietnamese restaurant for 3 in Houston, next Friday around dinner?", "expected": {"people": 3, "cuisine": "vietnamese", "city": "houston", "day": "friday", "time": "7 PM"}, "difficulty": "medium"},
    {"id": "s9_medium", "text": "I need a reservation at a Spanish tapas bar for 6 people in Atlanta next Saturday evening.", "expected": {"people": 6, "cuisine": "spanish", "city": "atlanta", "day": "saturday", "time": "7 PM"}, "difficulty": "medium"},
    {"id": "s10_medium", "text": "Book me at a Lebanese restaurant for 4 in Chicago, this Thursday night please.", "expected": {"people": 4, "cuisine": "lebanese", "city": "chicago", "day": "thursday", "time": "7 PM"}, "difficulty": "medium"},
    {"id": "s11_medium", "text": "Looking for a Turkish restaurant in Los Angeles for 5 people on Friday evening.", "expected": {"people": 5, "cuisine": "turkish", "city": "los angeles", "day": "friday", "time": "7 PM"}, "difficulty": "medium"},
    {"id": "s12_medium", "text": "Find me a Nordic restaurant for 2 in Seattle next Monday at 7:30 PM.", "expected": {"people": 2, "cuisine": "nordic", "city": "seattle", "day": "monday", "time": "7:30 PM"}, "difficulty": "medium"},
    {"id": "s13_medium", "text": "Can you reserve a Moroccan place for 4 in Austin this coming Wednesday around 8 PM?", "expected": {"people": 4, "cuisine": "moroccan", "city": "austin", "day": "wednesday", "time": "8 PM"}, "difficulty": "medium"},
    {"id": "s14_medium", "text": "I need a reservation at an Italian fine dining for 6 in San Francisco next Friday night.", "expected": {"people": 6, "cuisine": "italian", "city": "san francisco", "day": "friday", "time": "8 PM"}, "difficulty": "medium"},

    {"id": "s1_hard", "text": "Book me a sushi restaurant in a major city for the weekend.", "expected": {"people": 2, "cuisine": "sushi", "city": "san francisco", "day": "saturday", "time": "7 PM"}, "difficulty": "hard"},
    {"id": "s2_hard", "text": "Find a Japanese place in Los Angeles for 3 people tomorrow night.", "expected": {"people": 3, "cuisine": "japanese", "city": "los angeles", "day": "tomorrow", "time": "7 PM"}, "difficulty": "hard"},
    {"id": "s3_hard", "text": "Book a romantic dinner for two this weekend, somewhere nice with good pasta.", "expected": {"people": 2, "cuisine": "italian", "city": "san francisco", "day": "saturday", "time": "8 PM"}, "difficulty": "hard"},
    {"id": "s4_hard", "text": "Get me into a trendy spot for brunch with 4 friends this weekend.", "expected": {"people": 5, "cuisine": "brunch", "city": "san francisco", "day": "sunday", "time": "11 AM"}, "difficulty": "hard"},
    {"id": "s5_hard", "text": "I'm celebrating my birthday next week with 7 guests. Find somewhere special with good wine.", "expected": {"people": 8, "cuisine": "wine bar", "city": "san francisco", "day": "saturday", "time": "7 PM"}, "difficulty": "hard"},
    {"id": "s6_hard", "text": "Need something upscale for a client dinner, party of 4, somewhere downtown.", "expected": {"people": 4, "cuisine": "fine dining", "city": "new york", "day": "thursday", "time": "7:30 PM"}, "difficulty": "hard"},
    {"id": "s7_hard", "text": "Find me a spot for a casual lunch with colleagues, but also good for dinner later.", "expected": {"people": 6, "cuisine": "american", "city": "chicago", "day": "tuesday", "time": "12 PM"}, "difficulty": "hard"},
    {"id": "s8_hard", "text": "Book something for a food critic reviewing new restaurants this month.", "expected": {"people": 2, "cuisine": "fusion", "city": "new york", "day": "wednesday", "time": "8 PM"}, "difficulty": "hard"},
    {"id": "s9_hard", "text": "I want an experience, not just a meal. Make it memorable for the team.", "expected": {"people": 8, "cuisine": "tasting menu", "city": "san francisco", "day": "friday", "time": "7 PM"}, "difficulty": "hard"},
    {"id": "s10_hard", "text": "Find the best Thai restaurant in town, really high-end, for 3.", "expected": {"people": 3, "cuisine": "thai", "city": "los angeles", "day": "saturday", "time": "7 PM"}, "difficulty": "hard"},
    {"id": "s11_hard", "text": "Surprise me with a hidden gem restaurant that's popular but hard to get into.", "expected": {"people": 2, "cuisine": "contemporary", "city": "austin", "day": "friday", "time": "8 PM"}, "difficulty": "hard"},
    {"id": "s12_hard", "text": "I need a reservation at a Michelin-star restaurant if possible, for 4.", "expected": {"people": 4, "cuisine": "michelin", "city": "new york", "day": "saturday", "time": "8 PM"}, "difficulty": "hard"},
    {"id": "s13_hard", "text": "Book me somewhere that does great cocktails and modern cuisine for 3.", "expected": {"people": 3, "cuisine": "modern", "city": "miami", "day": "friday", "time": "7:30 PM"}, "difficulty": "hard"},
    {"id": "s14_hard", "text": "Find a restaurant perfect for a marriage proposal for two, very romantic.", "expected": {"people": 2, "cuisine": "romantic", "city": "san francisco", "day": "saturday", "time": "8 PM"}, "difficulty": "hard"},
]

# Sample scenarios to get balanced representation
def sample_scenarios(all_scenarios, per_difficulty=6, seed=42):
    random.seed(seed)
    easy = [s for s in all_scenarios if s["difficulty"] == "easy"]
    medium = [s for s in all_scenarios if s["difficulty"] == "medium"]
    hard = [s for s in all_scenarios if s["difficulty"] == "hard"]

    selected = (
        random.sample(easy, min(per_difficulty, len(easy))) +
        random.sample(medium, min(per_difficulty, len(medium))) +
        random.sample(hard, min(per_difficulty, len(hard)))
    )
    return selected

SCENARIOS = sample_scenarios(ALL_SCENARIOS, SCENARIO_SUBSET_SIZE, SEED)

lock = Lock()

def iso_now():
    return datetime.now().isoformat()

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                return json.load(f)
        except:
            return None
    return None

def save_checkpoint(completed_tasks, rows):
    checkpoint = {
        "completed_tasks": list(completed_tasks),
        "total_rows": len(rows),
        "timestamp": iso_now()
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f, indent=2)

def save_backup_results(rows):
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(RESULTS_FILE, index=False)

def calc_stats(vals: List[float]) -> Tuple[float, float, float]:
    if not vals:
        return 0.0, 0.0, 0.0
    m = statistics.mean(vals)
    if len(vals) <= 1:
        return m, 0.0, 0.0
    sd = statistics.stdev(vals)
    se = sd / math.sqrt(len(vals))
    ci = 1.96 * se
    return m, ci, se

def extract_json(text: str) -> Optional[str]:
    m = re.search(r'```json\s*(\{.*?\})\s*```', text, flags=re.DOTALL|re.IGNORECASE)
    if m:
        return m.group(1)
    start = None
    depth = 0
    for i, ch in enumerate(text):
        if ch == "{":
            if start is None:
                start = i
            depth += 1
        elif ch == "}" and start is not None:
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def parse_json(s: str) -> Optional[Dict]:
    try:
        return json.loads(s)
    except:
        try:
            return json.loads(s.replace("'", '"'))
        except:
            return None

def get_number(text: str) -> Optional[int]:
    m = re.search(r'\b([1-9][0-9]?)\b', text)
    if m:
        return int(m.group(1))
    words = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8}
    for w, val in words.items():
        if re.search(r'\b' + w + r'\b', text, flags=re.IGNORECASE):
            return val
    return None

def extract_json_fenced(text: str) -> Optional[str]:
    m = re.search(r'```json\s*(\{.*?\})\s*```', text, flags=re.DOTALL|re.IGNORECASE)
    if m:
        return m.group(1)
    m2 = re.search(r'```\s*(\{.*?\})\s*```', text, flags=re.DOTALL)
    if m2:
        return m2.group(1)
    return None

def detect_tool(text: str) -> Tuple[bool, Optional[Dict], Optional[str]]:
    if not text:
        return False, None, None
    jf = extract_json_fenced(text)
    if jf:
        p = parse_json(jf)
        return True, p, None if p else "parse_fail_fenced"
    j = extract_json(text)
    if j:
        p = parse_json(j)
        return True, p, None if p else "parse_fail_inline"
    if re.search(r'\b(reservation confirmed|booking confirmed|i have booked|i booked|confirmation id|confirmed|booked)\b', text, flags=re.IGNORECASE):
        return True, {"implied": True, "text": text}, None
    return False, None, None

def mock_book(params: Dict) -> Dict:
    cuisine = params.get("cuisine") or params.get("food") or "Restaurant"
    city = params.get("city") or "City"
    people = params.get("people") or params.get("party_size") or "unknown"
    time_ = params.get("time") or params.get("date") or ""
    restaurant_name = "{0} Place in {1}".format(str(cuisine).title(), str(city).title())
    confirmation = str(uuid.uuid4())[:12].upper()
    return {
        "status": "confirmed",
        "restaurant": restaurant_name,
        "confirmation_id": confirmation,
        "details": {"cuisine": cuisine, "city": city, "people": people, "time": time_},
        "timestamp": iso_now()
    }

def has_city(text: str, city: str) -> bool:
    return city.lower() in text.lower() if city else False

def has_cuisine(text: str, cuisine: str) -> bool:
    if not cuisine:
        return False
    t, c = text.lower(), cuisine.lower()
    if c in t:
        return True
    synonyms = {"sushi": ["japanese"], "japanese": ["sushi"], "vegan": ["plant-based"]}
    return any(s in t for s in synonyms.get(c, []))

def has_time(text: str, time_: str) -> bool:
    if not time_:
        return False
    if time_.lower() in text.lower():
        return True
    for digit in re.findall(r'\d{1,2}', time_):
        if re.search(r'\b' + digit + r'\b', text):
            return True
    return False

def has_day(text: str, day: str) -> bool:
    return day.lower() in text.lower() if day else False

def score_result(final: str, expected: Dict, tool_used: bool, tool_resp: Optional[Dict]) -> Tuple[int, Dict]:
    text = (final or "").lower()
    intent, params, tool, quality = 0, 0, 0, 0

    if re.search(r'\b(book|reserve|reservation|booking|table)\b', text):
        intent += 2
    if has_cuisine(text, expected.get("cuisine")):
        intent += 1
    if get_number(text):
        intent += 1
    if re.search(r'\b(confirm|booked|will)\b', text):
        intent += 1

    if expected.get("people"):
        if str(expected["people"]) in text or (get_number(text) == expected["people"]):
            params += 1
    if expected.get("cuisine") and has_cuisine(text, expected["cuisine"]):
        params += 1
    if expected.get("city") and has_city(text, expected["city"]):
        params += 1
    if expected.get("day") and has_day(text, expected["day"]):
        params += 1
    if expected.get("time") and has_time(text, expected["time"]):
        params += 1

    if tool_used:
        tool = 5 if (tool_resp and tool_resp.get("status") == "confirmed") else 3

    if re.search(r'\b(confirm|booked|reservation)\b', text):
        quality += 2
    if re.search(r'(restaurant:|people:|time:|confirmation:)', final, flags=re.IGNORECASE):
        quality += 1
    if all([expected.get(k) for k in ["city", "time", "people"]]) and has_city(text, expected["city"]) and str(expected["people"]) in text:
        quality += 1

    quality = min(quality, 5)
    total = min(20, intent + params + tool + quality)
    return total, {"intent": intent, "params": params, "tool": tool, "quality": quality}

def pass_rate(scores: List[int], threshold: int = 15) -> float:
    return sum(1 for s in scores if s >= threshold) / len(scores) if scores else 0.0

def call_api(client_oai, client_anthropic, client_gemini, model: str, msgs: List[Dict], temp: float, max_tok: int):
    retries = 10

    for attempt in range(retries):
        try:
            start = time.time()

            if "claude" in model.lower():
                sys_msg = next((m["content"] for m in msgs if m["role"] == "system"), "")
                user_msgs = [m for m in msgs if m["role"] != "system"]
                resp = client_anthropic.messages.create(
                    model=model, max_tokens=max_tok, temperature=temp,
                    system=sys_msg or "You are helpful.", messages=user_msgs
                )
                text = resp.content[0].text

            elif "gemini" in model.lower():
                gemini_msgs = []
                sys_msg = ""
                for m in msgs:
                    if m["role"] == "system":
                        sys_msg = m["content"]
                    else:
                        gemini_msgs.append({
                            "role": "user" if m["role"] == "user" else "model",
                            "parts": [{"text": m["content"]}]
                        })

                resp = client_gemini.generate_content(
                    contents=gemini_msgs,
                    generation_config=genai.types.GenerationConfig(
                        temperature=temp,
                        max_output_tokens=max_tok,
                    ),
                    system_instruction=sys_msg or "You are helpful."
                )
                text = resp.text if resp.text else "No response"

            else:
                resp = client_oai.chat.completions.create(
                    model=model, messages=msgs, temperature=temp, max_tokens=max_tok
                )
                text = resp.choices[0].message.content

            # ADD DELAY AFTER SUCCESSFUL API CALL
            time.sleep(API_DELAY_SECONDS)

            return text.strip(), time.time() - start, None

        except Exception as e:
            error_str = str(e).lower()

            if "429" in str(e) or "quota" in error_str or "rate" in error_str:
                wait_time = min(60 * (2 ** attempt), 600)
                print("RATE LIMIT - Waiting {0}s (Retry {1}/{2})".format(wait_time, attempt + 1, retries))
                time.sleep(wait_time)
            elif attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
            else:
                return None, None, str(e)

    return None, None, "Max retries exceeded"

def run_scenario(c_oai, c_anth, c_gemini, model: str, scenario: Dict, temp: float, max_tok: int, max_t: int, run: int, seed: int):
    sys = "You are an assistant handling restaurant reservations. If booking, respond with JSON in ```json``` tags: {\"action\":\"call_tool\", \"tool\":\"book_table\", \"params\":{...}}"
    msgs = [{"role": "system", "content": sys}, {"role": "user", "content": scenario["text"]}]

    hist = []
    lats = []
    tool_called = False
    tool_json = None
    tool_err = None
    tool_resp = None
    turns = 0

    for _ in range(max_t):
        turns += 1
        text, lat, err = call_api(c_oai, c_anth, c_gemini, model, msgs, temp, max_tok)

        if err or text is None:
            return {"error": err or "No response", "turn": turns}

        lats.append(lat)
        hist.append(text)

        called, parsed, perr = detect_tool(text)

        if called:
            tool_called = True
            tool_json = parsed
            tool_err = perr

            if parsed and any(parsed.get(k) for k in ["action", "tool", "params", "confirmed", "implied"]):
                params = parsed.get("params", {}) if isinstance(parsed.get("params"), dict) else {}
                if not params:
                    params = {k: scenario["expected"].get(k) for k in ["people", "cuisine", "city", "time", "day"]}

                tool_resp = mock_book(params)
                msgs.append({"role": "assistant", "content": text})
                msgs.append({"role": "user", "content": "TOOL_RESPONSE: " + json.dumps(tool_resp)})
                continue
            else:
                msgs.append({"role": "assistant", "content": text})
                break
        else:
            msgs.append({"role": "assistant", "content": text})

            if re.search(r'\b(how many|party|when|city|cuisine)\b', text, flags=re.IGNORECASE):
                reply = None
                if re.search(r'\b(how many|party|people)\b', text.lower()):
                    reply = str(scenario["expected"].get("people"))
                elif re.search(r'\b(when|time|day|date)\b', text.lower()):
                    reply = scenario["expected"].get("day") or scenario["expected"].get("time")
                elif re.search(r'\b(city|where)\b', text.lower()):
                    reply = scenario["expected"].get("city")
                elif re.search(r'\b(cuisine|food|type)\b', text.lower()):
                    reply = scenario["expected"].get("cuisine")

                if reply:
                    msgs.append({"role": "user", "content": reply})
                    continue

            if re.search(r'\b(confirm|booked|reservation)\b', text, flags=re.IGNORECASE):
                break

    final = "\n".join(hist)
    return {
        "final": final, "hist": hist, "lats": lats, "tool_called": tool_called,
        "tool_json": tool_json, "tool_err": tool_err, "tool_resp": tool_resp, "turns": turns
    }

def task_worker(args):
    model, run_idx, scene, temp, max_tok, max_t, seed, oai, anth, gemini = args
    try:
        res = run_scenario(oai, anth, gemini, model, scene, temp, max_tok, max_t, run_idx, seed)

        if res.get("error"):
            return None

        final = res["final"]
        score, breakdown = score_result(final, scene["expected"], res["tool_called"], res["tool_resp"])
        lat_avg = statistics.mean(res["lats"]) if res["lats"] else 0.0

        return {
            "model": model,
            "run": run_idx,
            "scenario": scene["id"],
            "score": score,
            "intent": breakdown["intent"],
            "params": breakdown["params"],
            "tool": breakdown["tool"],
            "quality": breakdown["quality"],
            "turns": res["turns"],
            "latency": lat_avg,
            "tool_used": res["tool_called"],
            "difficulty": scene.get("difficulty", "unknown"),
            "final_text": final
        }, model, scene["id"], score
    except Exception as e:
        return None

def main():
    random.seed(SEED)
    np.random.seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    checkpoint = load_checkpoint()
    if checkpoint:
        print("RESUMING FROM CHECKPOINT: {0} tasks completed\n".format(len(checkpoint.get("completed_tasks", []))))

    print("="*80)
    print("API SETUP")
    print("="*80 + "\n")

    oai_key = os.environ.get("OPENAI_API_KEY", "")
    anth_key = os.environ.get("ANTHROPIC_API_KEY", "")
    gemini_key = os.environ.get("GOOGLE_API_KEY", "")

    if not oai_key:
        oai_key = input("OpenAI API key (press Enter to skip): ").strip()
    if not anth_key:
        anth_key = input("Anthropic API key (press Enter to skip): ").strip()
    if not gemini_key:
        gemini_key = input("Google Gemini API key (press Enter to skip): ").strip()

    oai = OpenAI(api_key=oai_key) if oai_key else None
    anth = Anthropic(api_key=anth_key) if anth_key else None
    gemini = None

    if gemini_key and HAS_GEMINI:
        genai.configure(api_key=gemini_key)
        try:
            gemini = genai.GenerativeModel("gemini-2.5-flash")
            print("Gemini 2.5 Flash initialized")
        except:
            try:
                gemini = genai.GenerativeModel("gemini-2.0-flash")
                print("Gemini 2.0 Flash initialized")
            except:
                gemini = genai.GenerativeModel("gemini-1.5-flash")
                print("Gemini 1.5 Flash initialized")

    if not oai and not anth and not gemini:
        print("ERROR: At least one API key is required")
        return

    active_models = []
    for m in MODELS:
        if "gpt" in m.lower() and oai:
            active_models.append(m)
        elif "claude" in m.lower() and anth:
            active_models.append(m)
        elif "gemini" in m.lower() and gemini:
            active_models.append(m)

    print("Active models: {0}".format(", ".join(active_models)))

    if not active_models:
        print("ERROR: No models available")
        return

    print("\n" + "="*80)
    print("BENCHMARK EXECUTION")
    print("="*80 + "\n")

    tasks = []
    for model in active_models:
        for run in range(1, N_RUNS + 1):
            for scene in SCENARIOS:
                tasks.append((model, run, scene, TEMPERATURE, MAX_TOKENS, MAX_TURNS, SEED, oai, anth, gemini))

    total_tasks = len(tasks)
    print("Total tasks: {0}".format(total_tasks))
    print("Workers: {0}".format(WORKERS))
    print("API delay per call: {0}s".format(API_DELAY_SECONDS))
    print("Estimated runtime: {0}-{1} minutes\n".format(
        int(total_tasks * API_DELAY_SECONDS / 60 * 0.8),
        int(total_tasks * API_DELAY_SECONDS / 60 * 1.2)
    ))

    rows = []
    summary_dict = {}
    timestamp = iso_now()
    completed_tasks = set()

    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = {executor.submit(task_worker, t): t for t in tasks}
        completed = 0

        for future in as_completed(futures):
            result = future.result()
            completed += 1

            if result:
                row, m, s, score = result
                with lock:
                    rows.append(row)
                    task_id = "{0}_{1}_{2}".format(m, s, row.get("run", 0))
                    completed_tasks.add(task_id)
                    key = (m, s)
                    summary_dict.setdefault(key, []).append(score)

                pct = (completed / total_tasks) * 100
                print("[{0:4d}/{1}] ({2:5.1f}%) {3:30s} | {4:15s} | Score: {5:2d}".format(
                    completed, total_tasks, pct, m, s, score))

                if completed % 20 == 0:
                    save_checkpoint(completed_tasks, rows)
                    save_backup_results(rows)

    if not rows:
        print("\nERROR: No results generated")
        return

    print("\n" + "="*80)
    print("RESULTS PROCESSING")
    print("="*80 + "\n")

    df = pd.DataFrame(rows)
    results_csv = os.path.join(OUTPUT_DIR, "results.csv")
    df.to_csv(results_csv, index=False)
    print("Saved: {0}".format(results_csv))
    print("  Total records: {0}".format(len(df)))

    summary_rows = []
    for (model, scenario), scores in sorted(summary_dict.items()):
        m, ci, se = calc_stats(scores)
        std = statistics.stdev(scores) if len(scores) > 1 else 0.0
        p10 = pass_rate(scores, 10)
        p15 = pass_rate(scores, 15)

        summary_rows.append({
            "model": model,
            "scenario_id": scenario,
            "runs": len(scores),
            "mean_score": round(m, 3),
            "std": round(std, 3),
            "se": round(se, 3),
            "ci95_lower": round(m - ci, 3),
            "ci95_upper": round(m + ci, 3),
            "pass_at_10": round(p10, 3),
            "pass_at_15": round(p15, 3),
            "min": min(scores),
            "max": max(scores)
        })

    summ_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(OUTPUT_DIR, "summary.csv")
    summ_df.to_csv(summary_csv, index=False)
    print("Saved: {0}".format(summary_csv))
    print("  Total records: {0}".format(len(summ_df)))

    meta = {
        "benchmark_name": NAME,
        "benchmark_version": VERSION,
        "focus": FOCUS,
        "timestamp": timestamp,
        "models_tested": active_models,
        "total_models": len(active_models),
        "runs_per_scenario": N_RUNS,
        "total_scenarios": len(SCENARIOS),
        "scenario_breakdown": {
            "easy": len([s for s in SCENARIOS if s["difficulty"] == "easy"]),
            "medium": len([s for s in SCENARIOS if s["difficulty"] == "medium"]),
            "hard": len([s for s in SCENARIOS if s["difficulty"] == "hard"])
        },
        "total_test_cases": len(rows),
        "workers": WORKERS,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "max_turns": MAX_TURNS,
        "api_delay_seconds": API_DELAY_SECONDS,
        "seed": SEED,
        "optimization_notes": "Workshop-optimized version with reduced models and scenarios to avoid rate limiting"
    }
    meta_path = os.path.join(OUTPUT_DIR, "metadata.json")
    save_json(meta, meta_path)
    print("Saved: {0}".format(meta_path))
    print("")

    print("="*80)
    print("BENCHMARK COMPLETE")
    print("="*80)
    print("")
    print("Output Directory: {0}".format(OUTPUT_DIR))
    print("")
    print("Generated Files:")
    print("  - results.csv ({0} rows)".format(len(df)))
    print("  - summary.csv ({0} rows)".format(len(summ_df)))
    print("  - metadata.json")
    print("")
    print("SUMMARY BY MODEL")
    print("-"*80)
    summary_by_model = df.groupby("model")["score"].agg(["mean", "std", "min", "max"]).round(3)
    print(summary_by_model)
    print("")

    print("SUMMARY BY DIFFICULTY")
    print("-"*80)
    summary_by_diff = df.groupby("difficulty")["score"].agg(["mean", "std", "count"]).round(3)
    print(summary_by_diff)
    print("")
    print("="*80 + "\n")

    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("Checkpoint cleaned up after successful completion\n")

if __name__ == "__main__":
    main()